In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score, f1_score, log_loss, classification_report

from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet, Ridge
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from xgboost import XGBClassifier, XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer 
from sklearn.decomposition import PCA     #**

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingClassifier, StackingRegressor

from tqdm import tqdm  # Provides the progress of model running
import os
import warnings
warnings.filterwarnings('ignore')

In [5]:
cal = pd.read_csv("C:/Users/PGCP-AI/Downloads/predictCalorieExpenditure/train.csv", index_col=0) 
cal

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
id,,,,,,,,
0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,female,38,166.0,61.0,25.0,102.0,40.6,146.0
...,...,...,...,...,...,...,...,...
749995,male,28,193.0,97.0,30.0,114.0,40.9,230.0
749996,female,64,165.0,63.0,18.0,92.0,40.5,96.0
749997,male,60,162.0,67.0,29.0,113.0,40.9,221.0


In [7]:
X , y = cal.drop('Calories', axis=1), cal['Calories']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26)

In [ ]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = make_column_transformer(( ohe, make_column_selector(dtype_include=object)),     
                          remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")

scaler = StandardScaler().set_output(transform='pandas')
svm = SVC()
comps = np.arange(2,8)
scores = []

for c in tqdm(comps):
    prcomp = PCA(n_components=c).set_output(transform='pandas')
    pipe = Pipeline([('OHE',trans),('SCL',scaler),('PCA',prcomp),('SVM',svm)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    scores.append([c, accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns=['comps', 'accuracy_score' ])
df_scores.sort_values(['accuracy_score', 'comps'], ascending=[False,True])

In [ ]:
scaler = StandardScaler().set_output(transform='pandas')
svm = SVC()
prcomp = PCA().set_output(transform='pandas')
pipe = Pipeline([('OHE',trans),('SCL',scaler),('PCA',prcomp),('SVM',svm)])

params = {'PCA__n_components': np.arange(2,8)}
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=26)
gcv = GridSearchCV(estimator=pipe, param_grid=params, cv=kfold, n_jobs=-1)
gcv.fit(X, y)
gcv.best_params_, gcv.best_score_

In [ ]:
tst = pd.read_csv("C:/Users/PGCP-AI/Downloads/predictCalorieExpenditure/test.csv") 
tst